# Model 14 — Mevcut As-of Feature Genişletme: Ders Kitabı Notebooku

Model 10'un test-dışı rolling-origin ölçümünde dört adayın da M-2 persistence
baseline'ını geçemediği (`pm_rapor_nowcast_rolling_origin.md`) ve Model 11'in
"mevcut bilgi temsilleri altında saptanabilir öngörü becerisi yoktur" hükmü
sonrası proje sahibi model performansını artırma odaklı çalışma talebinde
bulundu. Bu notebook, o talebe verilen ilk yanıt olan **Model 14**'ü açıklar:
Model 07'nin haftalık snapshot'ında kesit tarihinde zaten bilinen ama Model
09'un 10 feature'ında **kullanılmayan** kolonlardan küçük, domain-gerekçeli,
sızıntısız bir 4-feature ailesi eklenir ve aynı sıkı rolling-origin protokolü
(50 origin, 2 ay embargo, sabit seed, Model 10 ile birebir terfi kapısı)
tekrarlanır.

Ön-kayıt: `prompts/veri/43_model14_mevcut_asof_feature_genisletme_onkayit.md`
(commit `8a2e13a`, Rota-2 tarafından denetlendi ve Bölüm 9 ile düzeltildi —
bkz. aşağıda Bölüm 3). Bu notebook **sonuç görüldükten sonra** yazılmıştır;
ön-kayıt ise sonuç görülmeden önce kilitlenmişti — bu sıra N7/N12
falsifikasyon disiplini için önemlidir.

## Okuma hedefleri

- Model 14'ün neden başlatıldığını ve neyi test ettiğini anlamak.
- Kontrol (10 feature) ile test (14 feature) kolunun nasıl kurulduğunu ve
  kontrol kolunun Model 10 ile neden **bit-eşit** olması gerektiğini görmek.
- Ön-kayıt sonrası ortaya çıkan bir denetim düzeltmesini (Bölüm 9) ve neden
  "committed dosya" yerine "aynı süreçte canlı kod yolu" karşılaştırmasının
  daha sağlam olduğunu kavramak.
- Terfi kapısının (a-d) dört koşulunu ve neden hiçbirinin tek başına yeterli
  olmadığını kavramak.
- Nihai kararı (`TERFI_ADAYI_BULUNDU` / `SINYAL_YOK_14_FEATURE`) ve bunun
  arkasındaki sayısal gerekçeyi okuyabilmek.
- Bu çalışmanın sınırlılıklarını ve sonraki karar noktalarını görmek.

## 1. Bağlam: neden Model 14?

Aşama B'nin aktif hedefi (K9/K10), `noter_devir_otomobil_adet` hacim
serisinin bir sonraki takvim ayına göre üç sınıflı (`down/stable/up`, ±%5
sabit bant) yönüdür. Model 09, Model 07'nin 34 kolonluk haftalık
snapshot'ından yalnızca 10 feature seçip dört sabit aday (2 lojistik
regresyon + sığ Random Forest + sığ HistGradientBoosting) denemişti. Model
10 bu 10 feature'ı 50 test-dışı origin üzerinde (2021-03..2025-04, her
origin'de 2 ay embargo, yeniden fit) sınadı: **hiçbir aday M-2 persistence
baseline'ını terfi kapısından geçecek şekilde yenemedi**. Model 11, farklı
bilgi tavanlarını (lag yapısı, stable bant, oracle tavanları) tarayıp aynı
hükmü doğruladı.

Proje sahibinin "performans metriklerinin çok altındayız" talimatı üzerine
Model 14, en ucuz ve en az riskli iyileştirme yolunu dener: **yeni bir veri
kaynağı eklemeden**, zaten toplanmış ve zaten sızıntısız olan snapshot
kolonlarından, Model 09'da kullanılmamış küçük bir alt-küme seçmek.

In [1]:
import json
from pathlib import Path
import pandas as pd

repo = Path.cwd()
if not (repo / "data").exists():
    repo = repo.parent
model_dir = repo / "data" / "processed" / "model"

ozet = json.loads((model_dir / "model_14_mevcut_asof_feature_genisletme_ozet.json").read_text(encoding="utf-8"))

print(f"Karar: {ozet['karar']}")
print(f"Kontrol kolu feature sayisi: {ozet['kontrol_kolu']['feature_sayisi']}")
print(f"Test kolu feature sayisi: {ozet['test_kolu']['feature_sayisi']}")
print(f"Yeni feature ailesi: {ozet['yeni_featurelar']}")
print(f"Origin sayisi (her iki kol): {ozet['test_kolu']['origin_sayisi']}")
print(f"Degerlendirme ay araligi: {ozet['test_kolu']['degerlendirme_ay_araligi']}")
print(f"Kilitli test disarida birakilan satir sayisi: {ozet['kilitli_test_disarida_birakilan_satir_sayisi']}")
print(f"Test durumu: {ozet['test']}")
print(f"Calisma suresi (sn): {ozet['calisma_suresi_saniye']}")

Karar: SINYAL_YOK_14_FEATURE
Kontrol kolu feature sayisi: 10
Test kolu feature sayisi: 14
Yeni feature ailesi: ['usdtry_orta_std', 'tuketici_guven_endeksi_lag2ay', 'odmd_otomobil_adet_lag2ay', 'reel_politika_faizi_lag2ay']
Origin sayisi (her iki kol): 50
Degerlendirme ay araligi: ['2021-03', '2025-04']
Kilitli test disarida birakilan satir sayisi: 57
Test durumu: 2025-07..2026-06 ACILMADI_KILITLI
Calisma suresi (sn): 73.4


## 2. İki kollu tasarım: kontrol (10) vs test (14)

Kontrol kolu Model 09/10'un 10 feature'ıyla **birebir aynıdır** — hiçbir
şey değişmez. Test kolu bu 10'a, aşağıdaki tabloda özetlenen 4 yeni
feature'ı ekler. Üç tanesi Model 07 snapshot'ında zaten hazır (ham geçiş);
dördüncüsü (`reel_politika_faizi_lag2ay`) iki zaten-M-2-gecikmeli kolonun
farkından türetilir — bu yüzden gecikme/bilgi-zamanı kuralı bozulmaz.

In [2]:
yeni_feature_tablosu = pd.DataFrame([
    {
        "feature": "usdtry_orta_std",
        "tur": "ham gecis",
        "bilgi_gecikmesi": "yok (cari ay ici, cut-off'a kadar)",
        "domain_gerekce": "Kur ici-ay oynakligi: ithal maliyet belirsizligi/fiyatlama tereddudu proxy'si",
    },
    {
        "feature": "tuketici_guven_endeksi_lag2ay",
        "tur": "ham gecis",
        "bilgi_gecikmesi": "M-2",
        "domain_gerekce": "Talep tarafi dogrudan duyarlilik gostergesi",
    },
    {
        "feature": "odmd_otomobil_adet_lag2ay",
        "tur": "ham gecis",
        "bilgi_gecikmesi": "M-2",
        "domain_gerekce": "Yeni arac arzi/ikamesi; ikinci el talebini dolayli etkileyen dissal arz sinyali",
    },
    {
        "feature": "reel_politika_faizi_lag2ay",
        "tur": "turetilmis = politika_faizi_lag2ay - tufe_yillik_degisim_lag2ay",
        "bilgi_gecikmesi": "M-2 (iki bilesen de M-2)",
        "domain_gerekce": "Enflasyondan arindirilmis yaklasik reel faiz; parasal sikilik/gevseklik rejimi",
    },
])
yeni_feature_tablosu

,feature,tur,bilgi_gecikmesi,domain_gerekce
0,usdtry_orta_std,ham gecis,"yok (cari ay ici, cut-off'a kadar)",Kur ici-ay oynakligi: ithal maliyet belirsizli...
1,tuketici_guven_endeksi_lag2ay,ham gecis,M-2,Talep tarafi dogrudan duyarlilik gostergesi
2,odmd_otomobil_adet_lag2ay,ham gecis,M-2,Yeni arac arzi/ikamesi; ikinci el talebini dol...
3,reel_politika_faizi_lag2ay,turetilmis = politika_faizi_lag2ay - tufe_yill...,M-2 (iki bilesen de M-2),Enflasyondan arindirilmis yaklasik reel faiz; ...


## 3. Determinizm doğrulaması: kontrol kolu gerçekten Model 10 mu?

Ön-kayıt (Bölüm 6, STOP_ONLY_IF madde 8) ilk yazıldığında kontrol kolunun
`data/processed/model/model_10_rolling_origin_ozet.json` adlı yerel dosyayla
bit-eşit olmasını istiyordu. Bu dosya **Git tarafından izlenmiyor**
(`.gitignore`), yani "committed" değil, sadece yerelde kalmış bir üretim
artefaktı. Denetimde bu netleştirildi ve ön-kayıt **Bölüm 9** ile düzeltildi:
referans artık statik bir dosya değil, **aynı Python sürecinde, aynı HEAD
kodundan canlı çağrılan** `model_10_rolling_origin_nowcast._rolling_tahminleri`
fonksiyonudur. Bu, hem daha taşınabilir (ortamdan ortama dosya taşımaya
gerek yok) hem de daha güçlü bir kanıt: kodun *şu an* Model 10'u tekrar
ettiğini gösterir, altı gün önce üretilmiş statik bir dosyayla eşleştiğini
değil.

Bu düzeltmenin pratik gerekçesini gösteren bir yan-bulgu: ilk denemede
(farklı bir Python kurulumu, farklı scikit-learn sürümü) iki lojistik
regresyon adayında küçük ama gerçek sayısal farklar bulunmuştu — kök neden,
bu ikincil kurulumun projenin asıl ortamı olan `.venv312`'den farklı bir
scikit-learn/scipy sürümü taşımasıydı (Random Forest/HistGradientBoosting ve
üç baseline bu duyarlılığı hiç göstermedi). Aynı-süreç canlı karşılaştırma,
bu tür ortam kaynaklı belirsizliği yapısal olarak ortadan kaldırır: hangi
ortamda çalıştırılırsa çalıştırılsın, kontrol kolu ile canlı Model 10
referansı **aynı anda, aynı yorumlayıcıda** üretilir.

In [3]:
dogrulama = ozet["kontrol_model10_kod_yolu_dogrulama"]
print("Referans:", dogrulama["referans"])
print("Kontrol satir sayisi:", dogrulama["satir_sayisi_kontrol"])
print("Canli Model10 referans satir sayisi:", dogrulama["satir_sayisi_referans"])
print("Birebir uyumlu mu:", dogrulama["birebir_uyumlu"])
print("Canli Model10 assertion denetimi:", dogrulama["model10_assertion_denetimi"])
assert dogrulama["birebir_uyumlu"], "Kontrol kolu canli Model10 kod yoluyla uyusmuyor"
print()
print("Train-only fit sayisi (kontrol):", ozet["kontrol_kolu"]["assertion_denetimi"])
print("Train-only fit sayisi (test):   ", ozet["test_kolu"]["assertion_denetimi"])

Referans: ayni_HEAD_ayni_surec_model10__rolling_tahminleri
Kontrol satir sayisi: 1400
Canli Model10 referans satir sayisi: 1400
Birebir uyumlu mu: True
Canli Model10 assertion denetimi: {'on_isleme_fit_sayisi': 50, 'model_fit_sayisi': 200}

Train-only fit sayisi (kontrol): {'on_isleme_fit_sayisi': 50, 'model_fit_sayisi': 200}
Train-only fit sayisi (test):    {'on_isleme_fit_sayisi': 50, 'model_fit_sayisi': 200}


## 4. Terfi kapısı: dört koşul, referans M-2 persistence

Model 10'dan **gevşetilmeyen** terfi kapısı, test kolundaki her aday için
dört koşulun **tümünü** ister:

1. **(a)** Eşli hareketli-blok bootstrap + Holm alt sınırı > 0 (H0
   reddedilmiş olmalı VE düzeltilmiş alt sınır pozitif olmalı)
2. **(b)** Nokta ΔMCC ≥ 0.05 (M-2 persistence'a göre)
3. **(c)** Nokta Δmacro-F1 > 0
4. **(d)** Yıl-dışı (leave-one-year-out) jackknife'ta işaret her yıl
   korunuyor

Hafta 1→4 tanısı (`hafta_tanisi`) hiçbir koşulda terfi gerekçesi değildir —
yalnızca teşhis amaçlıdır.

In [4]:
def terfi_tablosu(kol_ozet):
    satirlar = []
    for ad, v in kol_ozet["terfi_karari_ham"].items():
        k = v["kosullar"]
        satirlar.append({
            "aday": ad,
            "nokta_mcc": kol_ozet["genel_metrikler"][ad]["nokta"]["mcc"],
            "delta_mcc_holm_alt_sinir": kol_ozet["holm_aile_4_model"][ad]["delta_mcc_holm_alt_sinir"],
            "a_holm": k["a_holm_alt_sinir_pozitif"],
            "b_delta_mcc_ge_005": k["b_delta_mcc_en_az_005"],
            "c_delta_f1_pos": k["c_macro_f1_farki_pozitif"],
            "d_jackknife": k["d_jackknife_isaret_korunuyor"],
            "TERFI": v["terfi"],
        })
    return pd.DataFrame(satirlar).set_index("aday")

print("--- TEST KOLU (14 feature) ---")
terfi_test = terfi_tablosu(ozet["test_kolu"])
terfi_test

--- TEST KOLU (14 feature) ---


,nokta_mcc,delta_mcc_holm_alt_sinir,a_holm,b_delta_mcc_ge_005,c_delta_f1_pos,d_jackknife,TERFI
aday,,,,,,,
lojistik_l2_c01,0.088595,-0.202015,False,True,True,True,False
lojistik_l2_c1,0.004086,-0.275842,False,False,False,False,False
random_forest_sigin,-0.047693,-0.361648,False,False,False,False,False
hist_gradient_sigin,0.012459,-0.346754,False,False,False,False,False


In [5]:
print("--- KONTROL KOLU (10 feature, referans) ---")
terfi_tablosu(ozet["kontrol_kolu"])

--- KONTROL KOLU (10 feature, referans) ---


,nokta_mcc,delta_mcc_holm_alt_sinir,a_holm,b_delta_mcc_ge_005,c_delta_f1_pos,d_jackknife,TERFI
aday,,,,,,,
lojistik_l2_c01,-0.070046,-0.316205,False,False,False,False,False
lojistik_l2_c1,-0.030604,-0.333197,False,False,False,False,False
random_forest_sigin,-0.119273,-0.347714,False,False,False,False,False
hist_gradient_sigin,-0.109673,-0.408967,False,False,False,False,False


## 5. Sonuç yorumu

Test kolunda **hiçbir aday dört koşulun tümünü karşılamadı** — nihai karar
`SINYAL_YOK_14_FEATURE`'dır. En yakın aday `lojistik_l2_c01`: nokta ΔMCC ve
Δmacro-F1 pozitif ve eşiği aşıyor (b ve c sağlanıyor), yıl-dışı
jackknife'ta işaret her yıl korunuyor (d sağlanıyor) — ama Holm-düzeltilmiş
hareketli-blok bootstrap alt sınırı hâlâ derin negatif (yaklaşık -0,20),
yani H0 reddedilemiyor (a sağlanmıyor). Başka bir deyişle: **nokta tahmini
iyileşmiş görünüyor ama N=50 origin'lik örneklemde bu iyileşme istatistiksel
gürültüden güvenle ayrılamıyor.**

Kontrol kolunun kendi nokta MCC'leri Model 10 ile birebir aynıdır (Bölüm
3'te doğrulandı); aşağıdaki ikincil karşılaştırma yalnız **bilgilendirme
amaçlıdır**, terfi kapısının bir parçası DEĞİLDİR (ön-kayıt Bölüm 6):

In [6]:
ikincil = pd.DataFrame(ozet["ikincil_kontrol_test_delta_bilgi_amacli_terfi_kapisi_disinda"]).T
ikincil.columns = ["delta_mcc (test-kontrol)", "delta_macro_f1 (test-kontrol)"]
ikincil

,delta_mcc (test-kontrol),delta_macro_f1 (test-kontrol)
lojistik_l2_c01,0.158641,0.128507
lojistik_l2_c1,0.034690,0.025564
random_forest_sigin,0.071580,0.049079
hist_gradient_sigin,0.122132,0.080349


Dört adayın **hepsinde** test kolu kontrol kolundan nokta MCC/F1 olarak
daha iyi (delta'ların tümü pozitif) — bu, 4 feature'lık ailenin tamamen
anlamsız olmadığının kanıtı, ama terfi kapısının aradığı istatistiksel güven
eşiğine ulaşmıyor. Bu, N6/N13'teki "dürüst negatif bulgu" ilkesiyle
tutarlıdır: iyileşme yönü var ama güven aralığı sıfırı rahatça içeriyor.

## 6. Sınırlılıklar

- **Küçük örneklem:** 50 test-dışı origin, bağımsız ay sayısı olarak hâlâ
  azdır; Holm-düzeltilmiş bootstrap alt sınırları geniştir (bkz. Model
  10'un `ci_yari_genislik_saptama_gucu_gostergesi` alanı, ~0,2-0,3
  mertebesinde).
- **Yalnız DF-A:** DF-B (N=29, N<50) bu protokole hiç girmedi;
  doğrulayıcı değildir.
- **Sabit ±%5 bant, sabit hiperparametreler:** Hiçbir tarama yapılmadı —
  onkayit bunu bilinçli olarak yasakladı (çoklu-test şişmesi riskine
  karşı, N7).
- **Küçük feature ailesi:** Yalnız 4 yeni feature denendi; Model 07
  snapshot'ında hâlâ yaklaşık 25 kullanılmayan kolon var (bkz. onkayit
  Bölüm 2'deki seçim gerekçesi) — daha geniş bir tarama burada
  YAPILMADI.
- **Lojistik regresyon çözücü hassasiyeti (Bölüm 3):** `lbfgs` çözücüsü,
  scikit-learn/scipy sürümüne bağlı küçük (yaklaşık 0,01 MCC
  mertebesinde) farklar üretebiliyor. Bölüm 3'teki canlı-kod-yolu
  düzeltmesi bu riski kontrol kolu için yapısal olarak ortadan kaldırır
  (aynı süreçte aynı yorumlayıcı kullanılır); ama test kolunun kendi
  lojistik sonuçları yine de farklı bir ortamda tam olarak
  yeniden üretilmeyebilir.
- **Hafta tanısı terfi gerekçesi değildir** ve bu çalışmada da öyle
  kullanılmadı (yukarıdaki `hafta_tanisi` alanları yalnız teşhis
  amaçlıdır).

## 7. Sonraki karar noktaları (yalnız öneri — başlatılmadı)

- `lojistik_l2_c01`'in a-koşulundaki büyük Holm alt sınırı açığı (nokta
  ΔMCC yaklaşık +0,07 iken alt sınır yaklaşık -0,20), asıl darboğazın
  feature kalitesinden çok **örneklem büyüklüğü/varyans** olduğuna işaret
  ediyor — origin sayısı zamanla (yeni aylar geldikçe) doğal olarak
  artacaktır.
- Model 07 snapshot'ındaki kalan yaklaşık 25 kullanılmayan kolondan (ör.
  kur seviye/min/max, ODMD dışındaki arz göstergeleri) ayrı, küçük
  ön-kayıtlı turlar denenebilir — ama Model 14 ile AYNI anda değil
  (çoklu-test şişmesini önlemek için tek seferde tek aile).
- Model 11'in sunduğu üç seçenek (hedefi kapat / bilgi kümesini
  değiştir / ufuk-sınıf yeniden tanımla) hâlâ proje sahibinin kararını
  bekliyor; Model 14'ün negatif sonucu bu üç seçeneği ELEMEZ.
- Kilitli test (2025-07..2026-06) **kapalı kalmalıdır** — bu çalışma onu
  açmadı ve açmamalıdır.

Bu bölüm yalnızca öneridir; K10 madde 6 gereği hiçbir sonraki adım bu
notebook veya PM raporu tarafından başlatılmamıştır.